# Generating Synthetic Multi-Turn Test Data Along Dimensions

## Overview

Writing test cases manually is time-consuming and biased toward scenarios the author already knows about. But "ask an LLM for some test queries" is worse: it returns a handful of repetitive variations on the most obvious scenario, with poor coverage of the edge cases that actually break your system.

A better approach, and the one taught here, is **dimension-driven synthetic data generation**:

1. Start by writing down concrete **failure hypotheses** for your system.
2. Turn those hypotheses into **dimensions**: categories that capture the kinds of variation that matter.
3. **Hand-write 10-20 seed tuples** (one value from each dimension) so you understand the space yourself.
4. **Scale with two-step generation**: LLM produces more tuples, a separate prompt converts each tuple into a natural-language query.
5. Feed the generated cases into the evaluators from notebook 05.

This gives you deliberate coverage of the behaviors you want to probe, instead of unpredictable coverage of whatever the LLM felt like writing.

## What You'll Learn

1. How to turn failure hypotheses into dimensions for the travel booking agent
2. How to hand-write seed tuples to validate the dimension schema
3. The two-step tuple-to-query pattern and why it produces more diverse language than a single prompt
4. The trade-off between cross-product-then-filter and direct LLM generation of tuples
5. How to convert generated cases into Strands `Case` objects and DeepEval `ConversationalGolden` objects
6. How Strands `ExperimentGenerator` fits in as one of several tools (appendix)

## 1. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import asyncio, json, random, itertools
import nest_asyncio
nest_asyncio.apply()

import boto3

from strands_evals import Case, Experiment

# Judge / generator model
GENERATOR_MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# A small direct-Bedrock helper so we can call the model with a schema-constrained prompt.
_bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

def bedrock_converse(system: str, user: str, max_tokens: int = 2000) -> str:
    resp = _bedrock.converse(
        modelId=GENERATOR_MODEL_ID,
        system=[{'text': system}],
        messages=[{'role': 'user', 'content': [{'text': user}]}],
        inferenceConfig={'maxTokens': max_tokens, 'temperature': 0.7},
    )
    return resp['output']['message']['content'][0]['text']

print('Generator ready.')

## 2. Step 1: Write Failure Hypotheses

Start by listing concrete ways the travel booking assistant is likely to fail. These are **hypotheses**, not proven failures. We are deciding what to probe, not what has already broken. The dimensions we define next will target these hypotheses.

| # | Failure Hypothesis |
|---|--------------------|
| H1 | The agent **invents flights** that weren't returned by `search_flights`, especially under price pressure or when the user seems impatient. |
| H2 | The agent **skips confirmation** and books directly when the user's request sounds decisive. |
| H3 | The agent **forgets state** across many turns or when the user abruptly changes topic. |
| H4 | The agent **handles past or malformed dates** poorly (calls tools with invalid YYYY-MM-DD strings). |
| H5 | The agent **struggles with budget constraints** that require filtering or tradeoffs. |
| H6 | The agent **cancels the wrong booking** when the user is ambiguous about which booking to cancel. |
| H7 | The agent **breaks scope** when the user asks for unrelated help (Python code, weather, legal advice). |

These map naturally to our evaluator failure modes F1-F7 from notebook 05.

## 3. Step 2: Derive Dimensions From the Hypotheses

A **dimension** is a category that captures one type of variation you want to produce. Each dimension has a small, closed set of values. Every test case is a **tuple**: one value picked from each dimension.

For the travel booking agent, four dimensions give strong coverage of the hypotheses above:

| Dimension | Values | Probes Hypothesis |
|-----------|--------|-------------------|
| **IntentType** | `search`, `book`, `review`, `cancel`, `modify` | H2, H3, H6 |
| **Complexity** | `single_leg`, `multi_step` (flight + hotel), `pivot` (topic switch mid-session) | H3 |
| **UserMood** | `neutral`, `impatient`, `price_sensitive`, `uncertain` | H1, H2, H5 |
| **EdgeCase** | `none`, `past_date`, `invalid_city`, `ambiguous_ref`, `off_topic` | H4, H6, H7 |

The dimensions are the schema. The tuples are the data. The question we answer next: which tuples do we actually want to test?

In [ ]:
# The dimension schema, encoded as a plain dict so we can mix-and-match programmatically.
DIMENSIONS = {
    'IntentType': ['search', 'book', 'review', 'cancel', 'modify'],
    'Complexity': ['single_leg', 'multi_step', 'pivot'],
    'UserMood': ['neutral', 'impatient', 'price_sensitive', 'uncertain'],
    'EdgeCase': ['none', 'past_date', 'invalid_city', 'ambiguous_ref', 'off_topic'],
}

total_tuples = 1
for vals in DIMENSIONS.values():
    total_tuples *= len(vals)
print(f'Dimension space: {" x ".join(str(len(v)) for v in DIMENSIONS.values())} = {total_tuples} possible tuples')
print('Most of that space is valid, a few combinations are not (e.g. cancel + invalid_city has no natural meaning).')

## 4. Step 3: Hand-Write Seed Tuples

Before asking an LLM to generate hundreds of tuples, write 10-15 by hand. This forces you to confirm that:

- each dimension is well defined,
- the tuples actually correspond to realistic scenarios,
- the edge-case values produce meaningful test cases.

If you cannot write a plausible scenario for a tuple, drop it or refine the dimensions.

In [ ]:
# Step 3: hand-written seed tuples. Each is one realistic scenario we want to cover.
# The order of fields is: (IntentType, Complexity, UserMood, EdgeCase).

SEED_TUPLES = [
    # Core workflows
    ('search', 'single_leg', 'neutral', 'none'),
    ('book', 'multi_step', 'price_sensitive', 'none'),
    ('review', 'single_leg', 'neutral', 'none'),
    ('cancel', 'single_leg', 'uncertain', 'ambiguous_ref'),
    ('modify', 'multi_step', 'impatient', 'none'),

    # Complexity-driven
    ('book', 'pivot', 'uncertain', 'none'), # flight -> hotel mid-session
    ('search', 'multi_step', 'price_sensitive', 'none'), # flight + hotel with budget

    # Edge cases
    ('book', 'single_leg', 'impatient', 'past_date'),
    ('search', 'single_leg', 'neutral', 'invalid_city'),
    ('cancel', 'single_leg', 'impatient', 'ambiguous_ref'),

    # Adversarial
    ('book', 'single_leg', 'neutral', 'off_topic'), # user starts off-topic then pivots
]

for t in SEED_TUPLES:
    print(f' {t}')
print(f'\nTotal seed tuples: {len(SEED_TUPLES)}')

## 5. Step 4: Generate More Tuples With the LLM

Now we scale. Two generation approaches:

| Approach | How It Works | When to Use |
|----------|--------------|-------------|
| **Cross-product then filter** | Enumerate all combinations, ask an LLM to mark each as valid/invalid | When most combinations are valid; gives full coverage including rare combos |
| **Direct LLM generation** | Ask the LLM to produce tuples directly from the schema + seed examples | When many combinations are invalid or the space is huge; more realistic but misses rare combos |

We use direct LLM generation here, seeded with our hand-written tuples so the model understands the style.

In [ ]:
# Step 4: ask the LLM for more tuples, using the seeds as few-shot examples.

TUPLE_SYSTEM_PROMPT = (
    'You generate structured test-case tuples for a travel booking chatbot. '
    'You will be given a dimension schema, a few seed tuples, and a target count. '
    'Return JSON only: a list of objects, each with keys IntentType, Complexity, UserMood, EdgeCase. '
    'Favor diversity and edge cases. Do not repeat the seed tuples exactly.'
)

seeds_block = '\n'.join(
    f' - {{ "IntentType": "{t[0]}", "Complexity": "{t[1]}", "UserMood": "{t[2]}", "EdgeCase": "{t[3]}" }}'
    for t in SEED_TUPLES
)

user_prompt = (
    f'Dimension schema:\n'
    f' IntentType: {DIMENSIONS["IntentType"]}\n'
    f' Complexity: {DIMENSIONS["Complexity"]}\n'
    f' UserMood: {DIMENSIONS["UserMood"]}\n'
    f' EdgeCase: {DIMENSIONS["EdgeCase"]}\n\n'
    f'Seed tuples (already covered, do not repeat):\n{seeds_block}\n\n'
    'Generate 8 new tuples. Favor coverage of EdgeCase values that are under-represented '
    'in the seeds. Output JSON only.'
)

resp = bedrock_converse(TUPLE_SYSTEM_PROMPT, user_prompt)

# Robust JSON extraction
start = resp.find('[')
end = resp.rfind(']') + 1
generated_tuples_raw = json.loads(resp[start:end])
GENERATED_TUPLES = [(t['IntentType'], t['Complexity'], t['UserMood'], t['EdgeCase'])
                    for t in generated_tuples_raw]

for t in GENERATED_TUPLES:
    print(f' {t}')
print(f'\nTotal generated tuples: {len(GENERATED_TUPLES)}')

## 6. Step 5: Convert Tuples to Natural-Language Queries (Second Step)

Keeping tuple generation and query generation in **separate prompts** is the key trick. A single prompt that does both tends to produce repetitive phrasing, with every scenario reading like a variation of the same sentence. Splitting the two steps lets the query generator focus on realism without worrying about the underlying schema, and produces much more diverse language.

The query generator also emits a **goal** and an **expected assertion** for each tuple, so we can plug the cases directly into the evaluators from notebook 05.

In [ ]:
# Step 5: turn each tuple into a natural-language test case with input, goal, and success assertion.

QUERY_SYSTEM_PROMPT = (
    'You convert abstract test tuples into realistic test cases for a travel booking chatbot. '
    'For each tuple, produce JSON with keys: '
    ' name (short slug, lowercase, dashes), '
    ' input (the first user message - realistic, not generic), '
    ' goal (what the user wants to achieve), '
    ' expected_assertion (a single PASS/FAIL criterion a human reviewer could check). '
    'The expected_assertion MUST be a binary pass/fail statement, not a scale or a vague quality judgement. '
    'Output JSON only: a list of objects, one per tuple.'
)

ALL_TUPLES = SEED_TUPLES + GENERATED_TUPLES

tuples_block = '\n'.join(
    f' {i+1}. IntentType={t[0]}, Complexity={t[1]}, UserMood={t[2]}, EdgeCase={t[3]}'
    for i, t in enumerate(ALL_TUPLES)
)

resp = bedrock_converse(
    QUERY_SYSTEM_PROMPT,
    f'Tuples:\n{tuples_block}\n\nProduce one JSON object per tuple, in order. Output JSON only.',
    max_tokens=4000,
)

start = resp.find('[')
end = resp.rfind(']') + 1
cases_raw = json.loads(resp[start:end])

# Preview a handful
for obj, t in zip(cases_raw[:4], ALL_TUPLES[:4]):
    print(f'Tuple: {t}')
    print(f' name: {obj["name"]}')
    print(f' input: {obj["input"]}')
    print(f' goal: {obj["goal"]}')
    print(f' assertion: {obj["expected_assertion"]}')
    print()

print(f'Total generated cases: {len(cases_raw)}')

## 7. Step 6: Package as Strands `Case` Objects

The generated cases plug directly into the evaluator pipeline from notebook 05. Each `Case` carries the dimension tuple in `metadata`, so you can later slice the results, for example by pass rate broken down by `EdgeCase`.

In [ ]:
# Step 6: build Strands Case objects. Keep the tuple in metadata for later analysis.

strands_cases = []
for obj, t in zip(cases_raw, ALL_TUPLES):
    strands_cases.append(Case(
        name=obj['name'],
        input=obj['input'],
        expected_assertion=obj['expected_assertion'],
        metadata={
            'task_description': obj['goal'],
            'dimensions': {
                'IntentType': t[0],
                'Complexity': t[1],
                'UserMood': t[2],
                'EdgeCase': t[3],
            },
        },
    ))

print(f'Built {len(strands_cases)} Strands Case objects.')
print()
print('Coverage by EdgeCase:')
from collections import Counter
edge_counts = Counter(c.metadata['dimensions']['EdgeCase'] for c in strands_cases)
for k, v in edge_counts.most_common():
    print(f' {k:20s} {v}')

## 8. Step 7: Also Produce DeepEval `ConversationalGolden` Objects

The same tuples can be packaged as `ConversationalGolden` for DeepEval-based pipelines (see notebook 04). The binary `expected_assertion` maps naturally to DeepEval's `expected_outcome`.

In [ ]:
# Step 7: same tuples, DeepEval format.
from deepeval.dataset import ConversationalGolden

deepeval_goldens = []
for obj in cases_raw:
    deepeval_goldens.append(ConversationalGolden(
        scenario=obj['goal'],
        expected_outcome=obj['expected_assertion'],
        user_description='User from the generated tuple persona (see dimensions metadata).',
    ))

print(f'Built {len(deepeval_goldens)} ConversationalGolden objects.')

## 9. Appendix: Strands `ExperimentGenerator` as an Alternative Tool

Strands Evals also ships `ExperimentGenerator.from_context_async()`, which generates test cases directly from a free-form context description. It is convenient when you don't yet have clear failure hypotheses or dimensions and want a quick starting set.

Its limitation is the same as any "give me test queries" approach: coverage is what the LLM happened to write, not what you explicitly probed for. For workload-specific evaluation of production systems, the dimension-driven pattern above is more reliable. Use `ExperimentGenerator` for exploration, then move to dimensions once you know what you care about.

In [ ]:
# Appendix: ExperimentGenerator as a quick-starter (kept brief on purpose).
from strands_evals.generators import ExperimentGenerator

TRAVEL_AGENT_CONTEXT = (
    'Travel booking assistant with tools: search_flights, search_hotels, book_flight, book_hotel, '
    'get_bookings, cancel_booking. Multi-turn; should confirm before booking; stays on topic.'
)

gen = ExperimentGenerator(
    input_type=str, output_type=str,
    include_expected_output=True, include_expected_trajectory=True, include_metadata=True,
    max_parallel_num_cases=3,
)

quick_experiment = asyncio.get_event_loop().run_until_complete(
    gen.from_context_async(
        context=TRAVEL_AGENT_CONTEXT,
        task_description='Multi-turn travel booking assistant.',
        num_cases=3, num_topics=2,
    )
)

print(f'ExperimentGenerator produced {len(quick_experiment.cases)} quick cases:')
for c in quick_experiment.cases:
    print(f' {c.name}: {str(c.input)[:80]}')
print()
print('Use this as a starter. When you have clear failure hypotheses, switch to the dimension-driven approach above.')

## Next Steps

- Continue to **`04-11-07-tool-simulation.ipynb`** for LLM-powered tool simulation.
- Or skip to **`04-11-08-e2e-pipeline.ipynb`** to wire dimensions + simulation + custom binary evaluators together.